# Практика · Сумаризація> Теорія: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·> Домашнє завдання: [homework.html](homework.html)Зошит самодостатній: усе, що тут відбувається, пояснюється на місці, лекцію відкриватине обовʼязково.**Задача, яку ми розвʼязуємо.** У нас є тексти й людські резюме до них. Треба(1) побудувати систему, яка сама вибирає резюме, і (2) — це важливіше — **зрозуміти,чи вміє наша метрика взагалі відрізнити добре резюме від поганого**.Що зробимо:1. дістанемо справжні пари «текст → резюме» з бази встановлених пакетів;2. порахуємо **стелю витягу** — скільки з еталонного резюме взагалі є в тексті;3. напишемо **ROUGE з нуля** (готового пакета в системі немає) і звіримо його з `sklearn`;4. порівняємо пʼять способів вибрати речення, зокрема тупий «перше речення»;5. поставимо метриці грубу перевірку: чи відрізняє вона своє резюме від чужого;6. розберемо, звідки беруться нулі ROUGE-2;7. покажемо, що ROUGE не бачить вигаданого факту.> ⏱ Зошит нічого не навчає — лише рахує. Заміряно: **близько восьми секунд процесорного> часу** на чотирьох ядрах без відеокарти. За годинником вийде помітно більше —> хвилина й довше, — бо машину ділять з іншими програмами. Саме тому міряємо> `time.process_time()`, а не годинник; число друкується в кінці.> ⚠️ **Твої числа не збіжаться з тими, що в лекції, і це нормально.** Дані ми беремо> зі списку пакетів, встановлених **на твоїй машині**. У кожного він свій. Відтворюється> тут не значення, а **форма**: покриття по парах слів завжди приблизно вдвічі нижче за> покриття по словах, перше речення завжди в лідерах, нічиї ROUGE-2 завжди виявляються> двома нулями.

In [ ]:
# Фіксуємо кількість потоків ДО імпорту numpy: інакше потоки крутяться в очікуванні,
# і це очікування рахується як процесорний час — замір роздувається в десятки разів.
import os
for var in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[var] = '1'

import sys, re, math, json, random, subprocess, collections, statistics, time

started_at = time.process_time()      # процесорний час, а не годинник:
                                      # на завантаженій машині годинник бреше

print('Python  ', sys.version.split()[0])
import sklearn
print('sklearn ', sklearn.__version__)
print()
print('Зерна фіксуємо явно — жодного випадкового результату в цьому зошиті немає.')

## 1 · Звідки беремо даніПари «довгий текст → коротке резюме» зазвичай розмічають спеціально й дорого. Але однатака пара вже лежить на машині з Linux, і її ніхто не робив для нас.У кожного встановленого пакета є два текстові поля:* `%{DESCRIPTION}` — опис на кілька речень;* `%{SUMMARY}` — один рядок, що називає пакет.Обидва писала одна жива людина — супровідник пакета. Вона не думала про сумаризацію:просто мусила заповнити два поля так, щоб коротке чесно називало довге. Це і є еталон,здобутий без розмітки.Спершу перевіримо, чи взагалі є з чого читати. Якщо даних немає, зошит скаже целюдською мовою й зупиниться, а не впаде стеком на двадцятій клітинці.

In [ ]:
FIELDS = ('NAME', 'SUMMARY', 'DESCRIPTION')

def rpm_available():
    """Чи є на машині база пакетів rpm, з якої ми читаємо тексти."""
    try:
        done = subprocess.run(['rpm', '--version'], capture_output=True, text=True, timeout=20)
        return done.returncode == 0
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return False

if not rpm_available():
    print('Даних для цього зошита на цій машині немає.')
    print()
    print('Ми читаємо описи встановлених пакетів через програму rpm — вона є в Fedora,')
    print('RHEL, CentOS, openSUSE. Тут її немає, тож читати нема звідки.')
    print()
    print('Що можна зробити:')
    print('  · запустити зошит на машині з rpm;')
    print('  · або підставити свої пари «довгий текст → коротке резюме» у змінну pairs')
    print('    нижче: потрібен список словників з ключами name, summary, desc.')
    print()
    print('На Debian та Ubuntu подібні поля віддає dpkg-query -W -f, але цей шлях ми')
    print('НЕ перевіряли: машини з Debian у нас не було. Не вважай його робочим,')
    print('доки не перевіриш сам.')
    raise SystemExit(0)

print('rpm знайдено — читаємо описи пакетів.')

In [ ]:
def load_packages(min_words=20):
    """Читає з бази rpm усі пакети, чий опис має щонайменше min_words слів.

    Розділювачі \x1f (між полями) і \x1e (між записами) беремо тому, що в описах
    трапляються і коми, і переноси рядка, і будь-який звичайний роздільник
    поламався б на першому ж пакеті.
    """
    query_format = '\x1f'.join('%{' + f + '}' for f in FIELDS) + '\x1e'
    raw = subprocess.run(['rpm', '-qa', '--qf', query_format],
                         capture_output=True, text=True).stdout
    records = []
    for chunk in raw.split('\x1e'):
        if not chunk.strip():
            continue
        parts = [p.strip() for p in chunk.split('\x1f')]
        if len(parts) != len(FIELDS):
            continue
        name, summary, desc = parts
        if desc and desc != '(none)' and len(desc.split()) >= min_words:
            records.append({'name': name, 'summary': summary, 'desc': desc})
    # сортуємо за іменем, щоб порядок не залежав від того, як база віддала записи
    records.sort(key=lambda r: r['name'])
    return records

packages = load_packages()
print(f'пакетів з описом на 20+ слів: {len(packages)}')
print()
print('приклад запису:')
example = packages[0]
print('  name   :', example['name'])
print('  summary:', example['summary'])
print('  desc   :', example['desc'][:120].replace('\n', ' '), '…')

## 2 · Токенізація й поділ на реченняДалі нам треба вміти дві речі: різати текст на **слова** й різати його на **речення**.Слово беремо як послідовність латинських літер і цифр, усередині якої може стояти дефісабо апостроф (`open-source`, `don't`). Усе переводимо в нижній регістр, розділові знакивикидаємо — так робить і стандартний ROUGE.Речення ріжемо по крапці, знаку оклику чи питання, **після яких іде велика літера абоцифра**. Умова про велику літеру потрібна, щоб не розрізати `libmetalink is a MetalinkC library.` посеред скорочення на кшталт `e.g.`.

In [ ]:
WORD = re.compile(r"[a-z0-9]+(?:[-'][a-z0-9]+)*")

def words(text):
    """Текст → список слів у нижньому регістрі."""
    return WORD.findall(text.lower())

def sentences(text):
    """Текст → список речень. Розрив тільки перед великою літерою або цифрою."""
    flat = re.sub(r'\s+', ' ', text).strip()
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9])', flat)
    return [p.strip() for p in parts if p.strip()]

# Лишаємо тільки пари, придатні для задачі: є резюме і в описі є з чого вибирати.
pairs = [p for p in packages
         if p['summary'] and len(sentences(p['desc'])) >= 2]

print(f'пар «опис → резюме», придатних для роботи: {len(pairs)}')
print()
desc_len = [len(words(p['desc'])) for p in pairs]
summ_len = [len(words(p['summary'])) for p in pairs]
sent_cnt = [len(sentences(p['desc'])) for p in pairs]
print(f'опис  : медіана {statistics.median(desc_len)} слів · '
      f'середнє {statistics.mean(desc_len):.1f}')
print(f'речень: медіана {statistics.median(sent_cnt)} · '
      f'середнє {statistics.mean(sent_cnt):.2f} · максимум {max(sent_cnt)}')
print(f'резюме: медіана {statistics.median(summ_len)} слів · '
      f'середнє {statistics.mean(summ_len):.2f}')
print(f'стиснення за медіанами: '
      f'{statistics.median(desc_len) / statistics.median(summ_len):.1f}x')

## 3 · Стеля витягу: рахуємо її **до** того, як щось будуватиЄ два способи скоротити текст.* **Витяг** бере готові шматки з тексту й склеює їх. Жодного нового слова.* **Переказ** пише нові слова: може переставити, обʼєднати, узагальнити.Ми будуватимемо витяг — він простий, швидкий і не вміє брехати. Але перш ніж будувати,варто спитати: **а чи є в тексті те, що написала людина?** Якщо слів резюме в текстінемає, витяг їх не дістане — не тому, що поганий, а тому, що бере слова лише звідти.Порахуємо покриття для трьох випадків: окремі слова, пари сусідніх слів (біграми) ітрійки. Заразом порахуємо, у скількох пар покриття **нульове** — там ROUGE-n дасть нульбудь-якій системі світу.

In [ ]:
def ngram_set(tokens, n):
    """Множина всіх послідовностей із n слів поспіль."""
    return set(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

print(f'{"n":>2}  {"покриття":>9}  {"пар, де нуль неминучий":>24}')
coverage_by_n = {}
for n in (1, 2, 3):
    shares, doomed = [], 0
    for p in pairs:
        ref = ngram_set(words(p['summary']), n)
        src = ngram_set(words(p['desc']), n)
        if not ref:
            # резюме коротше за n слів: n-грам у нього немає, ROUGE-n там нуль завжди
            doomed += 1
            continue
        shares.append(len(ref & src) / len(ref))
        if not (ref & src):
            doomed += 1
    coverage_by_n[n] = statistics.mean(shares)
    print(f'{n:>2}  {statistics.mean(shares):9.4f}  '
          f'{doomed:>7} з {len(pairs)} = {doomed / len(pairs):.4f}')

print()
print('Читаємо так: окремі слова резюме майже всі є в тексті, а пари сусідніх слів —')
print('уже ні. Людина, скорочуючи текст, бере ті самі слова й ставить їх в іншому')
print('порядку. Слова зберігаються, порядок — ні.')

## 4 · Пишемо ROUGE з нуля`ROUGE` — родина метрик, які міряють, наскільки те, що видала система (**кандидат**),перетинається з тим, що написала людина (**еталон**).Нам потрібні три величини, і всі вони будуються з одного числа — **скільки шматківзбіглося**:* **повнота** = збіглося ÷ (усіх шматків еталона) — «скільки з потрібного ми сказали»;* **точність** = збіглося ÷ (усіх шматків кандидата) — «скільки зі сказаного було потрібне»;* **F** = 2 × точність × повнота ÷ (точність + повнота) — одне число замість двох.F — це гармонійне середнє. Воно відрізняється від звичайного тим, що жорстко караєперекіс: при точності 1.0 і повноті 0.0 звичайне середнє дало б 0.5, а F дає 0.Різновиди відрізняються тим, що вважати «шматком»:* **ROUGE-1** — окреме слово;* **ROUGE-2** — пара сусідніх слів;* **ROUGE-L** — найдовший спільний ланцюжок слів, що йдуть в обох текстах у тому самому  порядку, але **не обовʼязково поспіль**.Готового пакета `rouge_score` у системі немає, і це на краще: писавши метрику самі, мипобачимо, з чого саме складається число.

In [ ]:
def ngram_counts(tokens, n):
    """Лічильник n-грам: скільки разів кожна послідовність із n слів трапилась."""
    return collections.Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

def f_measure(matched, in_candidate, in_reference):
    """Гармонійне середнє точності й повноти. Нуль, якщо ділити нема на що."""
    if not matched or not in_candidate or not in_reference:
        return 0.0
    precision = matched / in_candidate
    recall = matched / in_reference
    return 2 * precision * recall / (precision + recall)

def rouge_n_parts(candidate, reference, n):
    """Повертає (збіглося, у кандидата, в еталона) для n-грам."""
    cand = ngram_counts(words(candidate), n)
    ref = ngram_counts(words(reference), n)
    # перетин лічильників бере мінімум по кожному ключу: слово, сказане двічі,
    # зараховується двічі лише якщо воно двічі є і в еталоні
    matched = sum((cand & ref).values())
    return matched, sum(cand.values()), sum(ref.values())

def rouge_n(candidate, reference, n):
    return f_measure(*rouge_n_parts(candidate, reference, n))

def lcs_length(a, b):
    """Довжина найдовшої спільної підпослідовності двох списків слів.

    Перебирати всі підпослідовності неможливо — їх 2 у степені довжини. Тому
    заповнюємо таблицю: у клітинці (i, j) — довжина відповіді для перших i слів
    одного списку й перших j слів другого. Кожна клітинка рахується з сусідніх.
    """
    previous_row = [0] * (len(b) + 1)
    for x in a:
        current_row = [0] * (len(b) + 1)
        for j, y in enumerate(b, start=1):
            if x == y:
                current_row[j] = previous_row[j - 1] + 1
            else:
                current_row[j] = max(previous_row[j], current_row[j - 1])
        previous_row = current_row
    return previous_row[-1]

def rouge_l(candidate, reference):
    cand, ref = words(candidate), words(reference)
    return f_measure(lcs_length(cand, ref), len(cand), len(ref))

def rouge_all(candidate, reference):
    """Три числа однією дією: ROUGE-1, ROUGE-2, ROUGE-L."""
    return (rouge_n(candidate, reference, 1),
            rouge_n(candidate, reference, 2),
            rouge_l(candidate, reference))

print('метрику написано')

### Перевіряємо себе двома способамиПерший — звіряємо підрахунок n-грам із бібліотечним. `CountVectorizer` зі `sklearn` умієрахувати біграми; якщо задати йому той самий шаблон слова, його лічильники мусятьзбігтися з нашими **точно**.Другий — звіряємо `lcs_length` із чесним перебором усіх підпослідовностей на короткихрядках. Перебір повільний, тому й потрібна таблиця, — але на пʼятьох словах віндозволений і дає незалежну відповідь.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from itertools import combinations

# --- перевірка 1: наші біграми проти бібліотечних
sample_texts = [p['desc'] for p in pairs[:200]]
vectorizer = CountVectorizer(lowercase=True,
                             token_pattern=r"[a-z0-9]+(?:[-'][a-z0-9]+)*",
                             ngram_range=(2, 2))
matrix = vectorizer.fit_transform(sample_texts)
names = vectorizer.get_feature_names_out()

for row, text in enumerate(sample_texts):
    ours = collections.Counter(' '.join(k) for k in ngram_counts(words(text), 2).elements())
    theirs = collections.Counter({names[col]: int(matrix[row, col])
                                  for col in matrix[row].nonzero()[1]})
    assert ours == theirs, f'біграми розійшлися на тексті {row}'
print('✅ наші біграми збігаються з CountVectorizer на', len(sample_texts), 'текстах')

# --- перевірка 2: наш LCS проти чесного перебору
def lcs_bruteforce(a, b):
    """Найдовша спільна підпослідовність через перебір усіх підмножин a."""
    best = 0
    for size in range(len(a), 0, -1):
        if size <= best:
            break
        for idx in combinations(range(len(a)), size):
            piece = [a[i] for i in idx]
            # чи є piece підпослідовністю b
            it = iter(b)
            if all(any(x == y for y in it) for x in piece):
                best = max(best, size)
                break
    return best

rng = random.Random(0)
alphabet = 'abcde'
for _ in range(300):
    a = [rng.choice(alphabet) for _ in range(rng.randint(1, 6))]
    b = [rng.choice(alphabet) for _ in range(rng.randint(1, 6))]
    assert lcs_length(a, b) == lcs_bruteforce(a, b), (a, b)
print('✅ наш LCS збігається з чесним перебором на 300 випадкових парах')

## 5 · Рахуємо ROUGE руками на одній паріВізьмімо конкретний пакет і подивімось на всі проміжні числа. Якщо `libmetalink` у тебене встановлений, візьмемо перший-ліпший пакет, чиє резюме має щонайменше чотири слова —розбір буде той самий, числа інші.

In [ ]:
by_name = {p['name']: p for p in pairs}
demo = by_name.get('libmetalink')
if demo is None:
    demo = next(p for p in pairs if len(words(p['summary'])) >= 4)
    print(f'пакета libmetalink немає, беремо {demo["name"]}\n')

reference = demo['summary']                    # що написала людина
candidate = sentences(demo['desc'])[0]         # що видала б найпростіша система

print('ЕТАЛОН  :', reference)
print('КАНДИДАТ:', candidate)
print()
print('слова еталона  :', words(reference))
print('слова кандидата:', words(candidate))
print()

for n, label in ((1, 'ROUGE-1'), (2, 'ROUGE-2')):
    matched, in_cand, in_ref = rouge_n_parts(candidate, reference, n)
    precision = matched / in_cand if in_cand else 0.0
    recall = matched / in_ref if in_ref else 0.0
    print(f'{label}')
    print(f'  шматків у кандидата: {in_cand} · в еталона: {in_ref} · збіглося: {matched}')
    print(f'  точність = {matched} / {in_cand} = {precision:.4f}')
    print(f'  повнота  = {matched} / {in_ref} = {recall:.4f}')
    print(f'  F        = {rouge_n(candidate, reference, n):.4f}')
    print()

chain = lcs_length(words(candidate), words(reference))
print('ROUGE-L')
print(f'  довжина спільного ланцюжка: {chain}')
print(f'  точність = {chain} / {len(words(candidate))} = {chain / len(words(candidate)):.4f}')
print(f'  повнота  = {chain} / {len(words(reference))} = {chain / len(words(reference)):.4f}')
print(f'  F        = {rouge_l(candidate, reference):.4f}')
print()
print('біграми еталона  :', [' '.join(k) for k in ngram_counts(words(reference), 2)])
print('біграми кандидата:', [' '.join(k) for k in ngram_counts(words(candidate), 2)])

## 6 · Пʼять способів вибрати одне реченняТепер — власне системи. Усі вони роблять одне: беруть з опису **одне** речення. Різницялише в тому, як обирають.* **lead-8 слів** — перші вісім слів опису, не зважаючи на межі речень. Найтупіше, що  можна написати.* **перше речення** — перше речення цілком.* **центроїд tf-idf** — речення, найближче до «середнього змісту» опису. Ідея: важливе  речення схоже на весь документ.* **TextRank** — речення, найбільш схоже на інші речення. Будуємо граф, де вузли — це  речення, а вага ребра — їхня схожість, і шукаємо найцентральніший вузол тим самим  способом, яким колись ранжували вебсторінки.* **оракул витягу** — речення, яке **заднім числом** дає найвищий ROUGE-1. Це не система:  підглядати в еталон не можна. Це **стеля** — найкраще, що взагалі здатен дати вибір  одного речення.

In [ ]:
def system_lead8(pkg):
    """Перші вісім слів опису — навмисне найтупіший рубіж."""
    return ' '.join(words(pkg['desc'])[:8])

def system_first_sentence(pkg):
    return sentences(pkg['desc'])[0]

def system_centroid(pkg):
    """Речення з найбільшою вагою слів, рідкісних усередині цього ж опису.

    Вага слова = скільки разів воно в описі, поділити на кількість речень, де воно
    трапляється. Ділимо суму на корінь довжини, щоб не перемагало найдовше речення
    просто через кількість слів.
    """
    parts = sentences(pkg['desc'])
    in_how_many_sentences = collections.Counter(
        word for s in parts for word in set(words(s)))
    in_whole_desc = collections.Counter(words(pkg['desc']))
    best, best_score = parts[0], -1.0
    for s in parts:
        tokens = words(s)
        if not tokens:
            continue
        score = sum(in_whole_desc[w] / (1 + in_how_many_sentences[w])
                    for w in set(tokens)) / len(tokens) ** 0.5
        if score > best_score:
            best_score, best = score, s
    return best

def system_textrank(pkg, iterations=20, damping=0.85):
    """Найцентральніше речення графа схожостей.

    Вага ребра — скільки спільних слів у двох речень, поділити на корінь добутку
    їхніх довжин (щоб довгі речення не були схожі на все підряд). Далі двадцять
    разів переливаємо «важливість» по ребрах, як у PageRank.
    """
    parts = sentences(pkg['desc'])
    if len(parts) < 2:
        return parts[0]
    bags = [set(words(s)) for s in parts]
    n = len(parts)
    weight = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j and bags[i] and bags[j]:
                weight[i][j] = len(bags[i] & bags[j]) / ((len(bags[i]) * len(bags[j])) ** 0.5)
    score = [1.0] * n
    for _ in range(iterations):
        updated = []
        for i in range(n):
            incoming = sum(weight[j][i] * score[j] / max(sum(weight[j]), 1e-9)
                           for j in range(n) if j != i)
            updated.append((1 - damping) + damping * incoming)
        score = updated
    return parts[max(range(n), key=lambda i: score[i])]

def system_oracle(pkg):
    """Найкраще речення за ROUGE-1 — можливе лише заднім числом."""
    return max(sentences(pkg['desc']), key=lambda s: rouge_n(s, pkg['summary'], 1))

SYSTEMS = [
    ('lead-8 слів', system_lead8),
    ('перше речення', system_first_sentence),
    ('центроїд tf-idf', system_centroid),
    ('TextRank', system_textrank),
    ('оракул витягу', system_oracle),
]

# рахуємо один раз і зберігаємо: далі ці ж числа знадобляться ще кілька разів
picked = {name: [fn(p) for p in pairs] for name, fn in SYSTEMS}
scored = {name: [rouge_all(text, p['summary']) for text, p in zip(picked[name], pairs)]
          for name, _ in SYSTEMS}

print(f'{"система":<18}{"ROUGE-1":>9}{"ROUGE-2":>9}{"ROUGE-L":>9}'
      f'{"слів":>7}{"точн-1":>9}{"повн-1":>9}')
for name, _ in SYSTEMS:
    triples = scored[name]
    lengths = [len(words(t)) for t in picked[name]]
    precisions, recalls = [], []
    for text, p in zip(picked[name], pairs):
        matched, in_cand, in_ref = rouge_n_parts(text, p['summary'], 1)
        precisions.append(matched / in_cand if in_cand else 0.0)
        recalls.append(matched / in_ref if in_ref else 0.0)
    print(f'{name:<18}'
          f'{statistics.mean(t[0] for t in triples):9.4f}'
          f'{statistics.mean(t[1] for t in triples):9.4f}'
          f'{statistics.mean(t[2] for t in triples):9.4f}'
          f'{statistics.mean(lengths):7.1f}'
          f'{statistics.mean(precisions):9.4f}'
          f'{statistics.mean(recalls):9.4f}')

### Наскільки цим числам можна віритиОдне середнє нічого не каже про стійкість. Повторимо замір на випадкових **половинах**корпусу й подивимось, наскільки гуляє результат.І одразу — окремий урок. Порахуємо купу двічі: спершу по пʼятьох половинах, потім посорока. Якщо друга купа помітно ширша за першу, то перша була **оптимістично вузькою**,і висновок, збудований на ній, стояв би на піску.

In [ ]:
def pile(values, repeats):
    """Мінімум і максимум середнього по випадкових половинах списку."""
    means = []
    for seed in range(repeats):
        chosen = random.Random(seed).sample(range(len(values)), max(len(values) // 2, 1))
        means.append(statistics.mean(values[i] for i in chosen))
    return min(means), max(means)

for metric_index, metric_name in ((0, 'ROUGE-1'), (1, 'ROUGE-2'), (2, 'ROUGE-L')):
    print(f'{metric_name}, купа середніх по випадкових половинах корпусу')
    print(f'{"система":<18}{"повне":>8}   {"5 половин":<18}{"40 половин":<18}')
    for name, _ in SYSTEMS:
        values = [t[metric_index] for t in scored[name]]
        low5, high5 = pile(values, 5)
        low40, high40 = pile(values, 40)
        print(f'{name:<18}{statistics.mean(values):8.4f}   '
              f'{low5:.4f}..{high5:.4f}     {low40:.4f}..{high40:.4f}')
    print()

print('Купа з пʼятьох повторів — НИЖНЯ оцінка розкиду, а не розкид. Якщо на ній')
print('тримається висновок, повторів має бути більше.')

### Два чесних сумніви до цієї таблиці**Перший.** Оракул ми обирали за ROUGE-1 — а колонки ROUGE-2 і ROUGE-L у нього тоді неоракульні: там стоїть просто те, що набрало обране за ROUGE-1 речення. Порахуймо оракулдля кожної метрики окремо й подивімось, наскільки відрізняється.**Другий.** Наш центроїд — евристика, і легко звинуватити нас у тому, що ми навмисневзяли слабкого суперника. Порівняння чесне тоді, коли зусилля однакове. Тому напишемо**класичний** центроїд — косинус між tf-idf-вектором речення й tf-idf-вектором усьогоопису — і подивімось, чи справа в реалізації.

In [ ]:
# --- сумнів 1: оракул за кожною метрикою окремо
def oracle_by_metric(pkg, metric_index):
    return max(sentences(pkg['desc']),
               key=lambda s: rouge_all(s, pkg['summary'])[metric_index])

print('оракул, обраний за різними метриками:')
print(f'{"обирали за":<14}{"ROUGE-1":>9}{"ROUGE-2":>9}{"ROUGE-L":>9}')
for metric_index, metric_name in ((0, 'ROUGE-1'), (1, 'ROUGE-2'), (2, 'ROUGE-L')):
    triples = [rouge_all(oracle_by_metric(p, metric_index), p['summary']) for p in pairs]
    print(f'{metric_name:<14}'
          f'{statistics.mean(t[0] for t in triples):9.4f}'
          f'{statistics.mean(t[1] for t in triples):9.4f}'
          f'{statistics.mean(t[2] for t in triples):9.4f}')
print()
print('Різниця мала, але вона є. Тобто «стеля витягу» — це не одне число, а число')
print('ДЛЯ ТІЄЇ МЕТРИКИ, за якою обирали.')

In [ ]:
# --- сумнів 2: класичний центроїд замість нашої евристики
def system_centroid_cosine(pkg):
    """Речення, tf-idf-вектор якого найближчий до вектора всього опису."""
    parts = sentences(pkg['desc'])
    n = len(parts)
    in_how_many = collections.Counter(word for s in parts for word in set(words(s)))

    def tfidf(counter):
        return {w: c * math.log(1 + n / (1 + in_how_many[w])) for w, c in counter.items()}

    doc_vec = tfidf(collections.Counter(words(pkg['desc'])))
    doc_norm = math.sqrt(sum(v * v for v in doc_vec.values())) or 1e-9
    best, best_cos = parts[0], -1.0
    for s in parts:
        tokens = words(s)
        if not tokens:
            continue
        sent_vec = tfidf(collections.Counter(tokens))
        sent_norm = math.sqrt(sum(v * v for v in sent_vec.values())) or 1e-9
        cosine = sum(v * doc_vec.get(w, 0.0) for w, v in sent_vec.items()) / (sent_norm * doc_norm)
        if cosine > best_cos:
            best_cos, best = cosine, s
    return best

cosine_scores = [rouge_n(system_centroid_cosine(p), p['summary'], 1) for p in pairs]
heuristic = statistics.mean(t[0] for t in scored['центроїд tf-idf'])
print(f'наша евристика       : {heuristic:.4f}')
print(f'класичний центроїд   : {statistics.mean(cosine_scores):.4f}')
print()
print('Різниці немає. Отже центроїд програє не через кривий код, а тому, що на')
print('коротких текстах його ідея не працює: він обирає найдовше речення, а задача')
print('вимагає найкоротшого.')

## 7 · Довжина видачі: чому саме F, а не повнотаНазва ROUGE розшифровується як `Recall-Oriented Understudy for Gisting Evaluation`, іслово `Recall-Oriented` там не випадкове: спершу метрику придумували **як повноту**.Подивімось, чому саме цього робити не можна. Візьмемо «систему», яка видає перші `k`слів опису, і поганяємо `k`.

In [ ]:
print(f'{"k":>4}{"точність":>11}{"повнота":>10}{"F":>10}')
f_by_k = {}
for k in (4, 6, 8, 12, 20, 40, 80):
    precisions, recalls, fs = [], [], []
    for p in pairs:
        head = ' '.join(words(p['desc'])[:k])
        matched, in_cand, in_ref = rouge_n_parts(head, p['summary'], 1)
        precision = matched / in_cand if in_cand else 0.0
        recall = matched / in_ref if in_ref else 0.0
        precisions.append(precision)
        recalls.append(recall)
        fs.append(f_measure(matched, in_cand, in_ref))
    f_by_k[k] = statistics.mean(fs)
    print(f'{k:>4}{statistics.mean(precisions):11.4f}{statistics.mean(recalls):10.4f}'
          f'{statistics.mean(fs):10.4f}')

best_k = max(f_by_k, key=f_by_k.get)
print()
print(f'Максимум F — на {best_k} словах. Він стоїть УСЕРЕДИНІ сітки, а не скраю,')
print('тож це справжній максимум, а не межа перебору.')
print(f'Медіана людського резюме — {statistics.median(summ_len)} слів: метрика вважає')
print('найкращою видачу, помітно довшу за людську.')

In [ ]:
# А тепер крайній випадок: «резюме», яке нічого не скоротило.
precisions, recalls = [], []
for p in pairs:
    matched, in_cand, in_ref = rouge_n_parts(p['desc'], p['summary'], 1)
    precisions.append(matched / in_cand if in_cand else 0.0)
    recalls.append(matched / in_ref if in_ref else 0.0)

print('увесь опис як «резюме»:')
print(f'  повнота ROUGE-1  = {statistics.mean(recalls):.4f}')
print(f'  точність ROUGE-1 = {statistics.mean(precisions):.4f}')
print(f'  для порівняння, стеля покриття по словах = {coverage_by_n[1]:.4f}')
print()
print('Повнота майже дорівнює стелі покриття. Тобто якщо суддя дивиться лише на')
print('повноту, виграшна стратегія — не скорочувати взагалі. Саме тому ROUGE')
print('звітують як F-міру, а число повноти без довжини видачі не має сенсу.')

## 8 · Найгрубіша перевірка: своє резюме проти чужогоДосі ми обговорювали, справедливі числа чи ні. Тепер поставмо метриці питання, на якевідповідь має бути очевидна:> чи ставить вона **своєму** резюме більше, ніж **чужому**?Дослід простий. Беремо перше речення опису пакета. Міряємо його ROUGE проти **власного**резюме цього пакета, а потім — проти резюме **випадкового іншого**. Порівнюємо.Це найслабша вимога, яку взагалі можна висунути до метрики якості. Вона не питає, чичисло велике; вона питає лише, чи воно більше для правильної відповіді, ніж для завідомонеправильної.

In [ ]:
rng = random.Random(0)
own_wins = {1: 0, 2: 0, 3: 0}
ties = {1: 0, 2: 0, 3: 0}
tie_both_zero = {1: 0, 2: 0, 3: 0}
comparisons = 0

for p in pairs:
    lead = sentences(p['desc'])[0]
    other = pairs[rng.randrange(len(pairs))]
    if other['name'] == p['name']:
        continue                     # порівнювати пакет сам із собою нецікаво
    own = rouge_all(lead, p['summary'])
    alien = rouge_all(lead, other['summary'])
    comparisons += 1
    for i, key in enumerate((1, 2, 3)):
        if own[i] > alien[i]:
            own_wins[key] += 1
        elif own[i] == alien[i]:
            ties[key] += 1
            if own[i] == 0.0:
                tie_both_zero[key] += 1

print(f'порівнянь: {comparisons}')
print()
print(f'{"метрика":<10}{"своє виграло":>14}{"нічия":>9}{"чуже виграло":>14}'
      f'{"нічиї, де обидва нулі":>24}')
for key, label in ((1, 'ROUGE-1'), (2, 'ROUGE-2'), (3, 'ROUGE-L')):
    alien_wins = comparisons - own_wins[key] - ties[key]
    share_zero = tie_both_zero[key] / ties[key] if ties[key] else 0.0
    print(f'{label:<10}{own_wins[key] / comparisons:14.4f}{ties[key] / comparisons:9.4f}'
          f'{alien_wins / comparisons:14.4f}'
          f'{tie_both_zero[key]:>13} = {share_zero:.4f}')

print()
print('Остання колонка — головна. Нічия в ROUGE-2 — це не «майже однакові числа»,')
print('а два точні нулі: метрика не помиляється, вона мовчить.')

## 9 · Анатомія нуляЗвідки беруться ці нулі? Спокуслива відповідь — «система погана». Розберімо.Поділимо всі нулі ROUGE-2 на дві купи:* **неминучі** — у **всьому описі** немає жодної біграми з резюме. Тут нуль дістала б  будь-яка система світу, зокрема й оракул;* **програш системи** — потрібна біграма в описі є, але перше речення її не взяло.

In [ ]:
zeros_total = 0
zeros_impossible = 0

for p in pairs:
    lead = sentences(p['desc'])[0]
    ref_bigrams = ngram_set(words(p['summary']), 2)
    src_bigrams = ngram_set(words(p['desc']), 2)
    hopeless = (not ref_bigrams) or not (ref_bigrams & src_bigrams)
    if rouge_n(lead, p['summary'], 2) == 0.0:
        zeros_total += 1
        if hopeless:
            zeros_impossible += 1

zeros_system = zeros_total - zeros_impossible
print(f'усього пар                              : {len(pairs)}')
print(f'нулів ROUGE-2 у першого речення         : {zeros_total} '
      f'= {zeros_total / len(pairs):.4f}')
print(f'  з них неминучих (витяг у принципі не може): {zeros_impossible} '
      f'= {zeros_impossible / zeros_total:.4f} від нулів')
print(f'  з них програш системи                     : {zeros_system}')
print()
print('Висновок: на цій частині корпусу ROUGE-2 видає ту саму цифру всім системам,')
print('тобто порівнювати їх там не може взагалі.')

### Де саме стоїть нуль — у поганого резюме чи в доброгоМожна припустити, що нулі ROUGE-2 просто збігаються з поганими резюме. Перевіримо:подивимось, як часто ROUGE-2 нульовий **при доброму ROUGE-1**.

In [ ]:
lead_scores = [(p, rouge_all(sentences(p['desc'])[0], p['summary'])) for p in pairs]

print(f'{"відбір":<18}{"пар":>7}{"з них ROUGE-2 = 0":>20}{"частка":>10}')
for threshold in (0.0, 0.3, 0.4, 0.5):
    chosen = [(p, s) for p, s in lead_scores if s[0] >= threshold]
    zeros = [(p, s) for p, s in chosen if s[1] == 0.0]
    label = 'усі пари' if threshold == 0.0 else f'ROUGE-1 >= {threshold:.2f}'
    print(f'{label:<18}{len(chosen):>7}{len(zeros):>20}'
          f'{len(zeros) / max(len(chosen), 1):10.4f}')

print()
print('Хвіст, який лишається, — найцікавіше в темі. Це пари, де кандидат каже ТЕ САМЕ,')
print('що еталон, іншими словами, а ROUGE-2 ставить нуль:')
print()
paraphrases = [(p, s) for p, s in lead_scores if s[0] >= 0.45 and s[1] == 0.0]
for p, s in paraphrases[:10]:
    print(f'  · {p["name"]}')
    print(f'    резюме: {p["summary"]}')
    print(f'    видало: {sentences(p["desc"])[0][:100]}')
    print(f'    ROUGE-1 = {s[0]:.4f}   ROUGE-2 = {s[1]:.4f}')
if not paraphrases:
    print('  (на твоєму наборі пакетів таких пар не знайшлося — спробуй поріг нижче)')

## 10 · TextRank проти першого речення: порівняння по кожному пакетуСереднє каже, хто попереду. Воно не каже, **чи різниця систематична** і в чому саме вонаполягає. Для цього треба порівнювати не середні, а кожен приклад окремо.

In [ ]:
first = [t[0] for t in scored['перше речення']]
other_names = ['lead-8 слів', 'центроїд tf-idf', 'TextRank']

print('«перше речення» проти інших, порівняння на кожному пакеті окремо')
print(f'{"суперник":<18}{"перше краще":>13}{"однаково":>11}{"суперник кращий":>18}')
for name in other_names:
    rival = [t[0] for t in scored[name]]
    wins = sum(1 for a, b in zip(first, rival) if a > b)
    losses = sum(1 for a, b in zip(first, rival) if a < b)
    draws = len(first) - wins - losses
    print(f'{name:<18}{wins / len(first):13.4f}{draws / len(first):11.4f}'
          f'{losses / len(first):18.4f}')

print()
same_choice = sum(1 for chosen, p in zip(picked['TextRank'], pairs)
                  if chosen == sentences(p['desc'])[0])
print(f'TextRank обрав ТЕ САМЕ перше речення в {same_choice} описах з {len(pairs)} '
      f'= {same_choice / len(pairs):.4f}')
print()
print('Ось звідки береться більшість нічиїх: у цих описах TextRank і рубіж lead —')
print('буквально та сама система, тільки одна з них рахує граф. Уся різниця')
print('збирається там, де TextRank ВІДХИЛЯЄТЬСЯ від початку тексту.')

## 11 · А чи не в тому річ, що описи короткі?Тут треба засумніватись у власному висновку. Медіана нашого опису — три речення, і втакому описі TextRank майже нема чого ранжувати. «Перше речення виграло» могло б бутитвердженням не про методи, а про те, що ми поставили граф у безвихідь.Слабкий результат слабкої системи нічого не доводить про задачу — він доводить лише, щосистемі не дали працювати. Перевіримо: повторимо замір на самих лише довгих описах.

In [ ]:
watched = ['перше речення', 'TextRank', 'центроїд tf-idf', 'оракул витягу']
for threshold in (2, 3, 4, 5, 6):
    keep = [i for i, p in enumerate(pairs) if len(sentences(p['desc'])) >= threshold]
    print(f'--- описи від {threshold} речень: {len(keep)} ---')
    if len(keep) < 30:
        print('    менше за 30 описів — на такому нічого не стверджуємо')
        continue
    for name in watched:
        values = [scored[name][i][0] for i in keep]
        low, high = pile(values, 40)
        print(f'    {name:<18}{statistics.mean(values):8.4f}   '
              f'купа 40 половин {low:.4f}..{high:.4f}')
    same = sum(1 for i in keep if picked['TextRank'][i] == sentences(pairs[i]['desc'])[0])
    covered = [len(ngram_set(words(pairs[i]['summary']), 1)
                   & ngram_set(words(pairs[i]['desc']), 1))
               / len(ngram_set(words(pairs[i]['summary']), 1))
               for i in keep if words(pairs[i]['summary'])]
    print(f'    TextRank обрав перше речення: {same / len(keep):.4f}   '
          f'покриття по словах: {statistics.mean(covered):.4f}')

print()
print('Читай два стовпчики поруч: «перше речення» і «TextRank». Якщо відрив між ними')
print('з довжиною тексту не зникає, то він не був наслідком коротких описів.')
print('А тепер подивись на «оракул витягу» окремо — він поводиться інакше, ніж усі.')

## 12 · Чого ROUGE не бачить зовсімУсе вище — способи метрики **занизити** оцінку правильному резюме. Тепер зворотний бік,і він небезпечніший.Головна вада сумаризації переказом зветься **галюцинація**: модель пише факт, якого втексті немає. Не помиляється в слові — **вигадує зміст**.Перевіримо, чи побачить це ROUGE. Еталон лишається тим самим, а кандидатів побудуємосамі: два правильні й два з вигаданим фактом. Це єдине місце в зошиті, де текст**написаний нами**, а не взятий із системи, — і саме тому, що нам потрібні кандидати звідомою наперед правдивістю.

In [ ]:
if demo['name'] == 'libmetalink':
    trials = [
        ('перше речення опису',        'libmetalink is a Metalink C library',   True),
        ('переказ своїми словами',     'A C library for reading Metalink files', True),
        ('ХИБНО: не та мова',          'Metalink library written in Java',       False),
        ('ХИБНО: не бібліотека',       'Metalink server written in C',           False),
    ]
    print('ЕТАЛОН:', reference)
    print()
    print(f'{"кандидат":<26}{"ROUGE-1":>9}{"ROUGE-2":>9}{"ROUGE-L":>9}{"правда?":>10}')
    for label, text, truthful in trials:
        r1, r2, rl = rouge_all(text, reference)
        print(f'{label:<26}{r1:9.4f}{r2:9.4f}{rl:9.4f}{("так" if truthful else "НІ"):>10}')
    print()
    print('Резюме, що називає НЕ ТУ мову програмування, дістає кращі числа за всіма')
    print('трьома метриками, ніж правильне перше речення опису. Причина проста: ROUGE')
    print('рахує збіг рядків і не знає, яке саме слово несе зміст. Заміна C на Java —')
    print('це рівно один незбіг, стільки ж, скільки коштував би будь-який артикль.')
else:
    print(f'Демонстрацію написано під пакет libmetalink, а в тебе взято {demo["name"]}.')
    print('Зроби те саме руками: візьми його резюме, зміни в ньому ОДНЕ змістовне')
    print('слово на неправильне — і подивись, наскільки впаде ROUGE. Здебільшого')
    print('не впаде майже ніяк, і це і є урок.')

## 13 · Що з усього цього випливаєСім правил, кожне з яких спирається на число вище.1. **Порахуй покриття до того, як щось навчати.** Одна хвилина роботи — і ти вже знаєш   стелю витягу й те, чи мовчатиме ROUGE-2 на частині даних.2. **Заміряй рубіж lead першим.** Перше речення. Якщо твоя модель його не бʼє, це привід   перевірити дані, а не крутити модель.3. **Звітуй F, а не повноту**, і називай довжину видачі поруч із числом.4. **Ніколи не звітуй сам ROUGE-2** як критерій: він не розрізняє чималу частину пар.5. **Порівнюй по парах, а не по середніх.** Середнє каже, хто попереду; порівняння на   кожному прикладі каже, чи це систематично.6. **Читай приклади очима.** Двадцять пар, прочитаних вручну, дають більше, ніж четвертий   знак середнього ROUGE.7. **Для переказу перевіряй фактологічність окремо** — ROUGE вигаданого факту не спіймає   ніколи.

In [ ]:
print(f'процесорний час усього зошита: {time.process_time() - started_at:.1f} с')
print()
print('Нагадування: усі числа вище заміряні на наборі пакетів ТВОЄЇ машини.')
print('На іншій машині вони будуть іншими — збігатися має форма, а не значення.')

## Завдання### 🟢 Рівень 1 — БазаДодай шосту систему: **останнє речення опису**. Заміряй її трьома метриками так само, якрешту, і встав рядок у таблицю розділу 6.**Зроблено, якщо:** у таблиці зʼявився рядок «останнє речення» з трьома числами, і тиможеш сказати одним реченням, чому воно виявилось гіршим (або кращим) за перше.### 🟡 Рівень 2 — ПлюсПобудуй витяг із **двох** речень замість одного: бери перше речення плюс те, яке додаєнайбільше нових слів. Заміряй ROUGE-1 і порівняй із витягом з одного речення.**Зроблено, якщо:** ти назвав, що саме зросло, а що впало — точність чи повнота — іпояснив, чому F зрушив саме в той бік, у який зрушив.### 🔴 Рівень 3 — ВикликПорахуй **оракул для ROUGE-2 окремо** — тобто обирай речення за ROUGE-2, а не заROUGE-1, — і порівняй із оракулом з розділу 6.Далі відповідай на питання: **чи можна називати одне число «стелею витягу»?** Перевірдві речі: (а) наскільки відрізняються стелі, обрані за різними метриками; (б) чи підійместелю дозвіл брати **два** речення замість одного.**Зроблено, якщо:** ти навів чотири числа (стеля за ROUGE-1 та за ROUGE-2, кожна дляодного й для двох речень) і сформулював, за яких умов слово «стеля» коректне, а за яких— ні.